# Tutorial :: Threats and opportunities in external data - the power of the news

**CONCERN**

You are working for a consultancy firm in charge of the Australian government's political image. In September 2021, the Australian government had a [high-profile problem with France](https://www.theguardian.com/australia-news/2021/sep/16/australia-nuclear-submarine-deal-contract-france-scrapped-defence-pact-us-uk) due to a deal to buy French submarines being called off. A report has already been generated with the titles of news items. However, your job as an analyst is to create a more thorough report taking into consideration additional information inside each news item.

In particular, your clients want to be aware of **threats** and **opportunities** suggested by the news.

1. **Q**uestion
2. **D**ata
3. **A**nalysis
4. **V**isualisation
5. **I**nsight

<img src="graphics/QDAVI_cycle_sm.png" width="50%" />

### 1. Question

How has the news affected the image of the Australian government?

**Tip:** You can combine web scraping and APIs

### 2. Data

We will use The Guardian API again.

**Tip:** Check the lecture and tutorial notebooks from Week 3 for information about how to call the guardian API

In [1]:
# Libraries for the analysis
import pandas as pd
import requests
import time
import json
from bs4 import BeautifulSoup

In [2]:
# Build a search URL
baseUrl = 'https://content.guardianapis.com/search?q=' # content search

searchString = "submarine"
section = "&section=world"
tag = "&tag=politics/politics"
fromDate = "&from-date=2021-09-01"
toDate = "&to-date=2021-11-30"
with open('private/guardian_key.txt', 'r') as file:
    key = file.read().strip()
api_key = f"&api-key={key}"

url = baseUrl+'"'+searchString+'"'+section+fromDate+toDate+api_key
print(url[:130])

https://content.guardianapis.com/search?q="submarine"&section=world&from-date=2021-09-01&to-date=2021-11-30&api-key=bff5b0ac-e949-


In [3]:
# Call the API
response = requests.get(url)
data = json.loads(response.content)
resp_data = data['response']
resp_data

{'status': 'ok',
 'userTier': 'developer',
 'total': 59,
 'startIndex': 1,
 'pageSize': 10,
 'currentPage': 1,
 'pages': 6,
 'orderBy': 'relevance',
 'results': [{'id': 'world/2021/oct/31/macron-accuses-australian-pm-of-lying-over-submarine-deal',
   'type': 'article',
   'sectionId': 'world',
   'sectionName': 'World news',
   'webPublicationDate': '2021-10-31T19:27:45Z',
   'webTitle': 'Macron accuses Australian PM of lying over submarine deal',
   'webUrl': 'https://www.theguardian.com/world/2021/oct/31/macron-accuses-australian-pm-of-lying-over-submarine-deal',
   'apiUrl': 'https://content.guardianapis.com/world/2021/oct/31/macron-accuses-australian-pm-of-lying-over-submarine-deal',
   'isHosted': False,
   'pillarId': 'pillar/news',
   'pillarName': 'News'},
  {'id': 'world/2021/oct/07/aukus-french-contractor-astonished-at-cancellation-of-australia-submarine-deal',
   'type': 'article',
   'sectionId': 'world',
   'sectionName': 'World news',
   'webPublicationDate': '2021-10-07T

In [4]:
def get_all_articles_for_response(response_json,full_url):
    total_pages = response_json['pages']
    total_articles = response_json['total']
    print(f"Fetching {total_articles} articles from {total_pages} pages...")
    all_articles = []
    page1_articles = response_json['results']
    all_articles.extend(page1_articles)
    print("Added articles for page: 1")
    
    for page in range(2,total_pages+1):
        print("Getting articles from API for page:",page)
        page_response = requests.get(full_url+f"&page={page}")
        page_data = page_response.json()['response']
        print("Processing results for page:",page_data['currentPage'])
        page_articles = page_data['results']
        print(f"Fetched {len(page_articles)} articles.")
        all_articles.extend(page_articles)
        print("Added articles for page:",page)
        print(f"Status: {len(all_articles)} articles.")
        time.sleep(1) # make sure we're not hitting the API to hard
    
    print(f"FINISHED: Fetched {len(all_articles)} articles.")
    return all_articles

results = get_all_articles_for_response(resp_data,url)

Fetching 59 articles from 6 pages...
Added articles for page: 1
Getting articles from API for page: 2
Processing results for page: 2
Fetched 10 articles.
Added articles for page: 2
Status: 20 articles.
Getting articles from API for page: 3
Processing results for page: 3
Fetched 10 articles.
Added articles for page: 3
Status: 30 articles.
Getting articles from API for page: 4
Processing results for page: 4
Fetched 10 articles.
Added articles for page: 4
Status: 40 articles.
Getting articles from API for page: 5
Processing results for page: 5
Fetched 10 articles.
Added articles for page: 5
Status: 50 articles.
Getting articles from API for page: 6
Processing results for page: 6
Fetched 9 articles.
Added articles for page: 6
Status: 59 articles.
FINISHED: Fetched 59 articles.


In [5]:
# # You can save the articles as a json file

# file_path = "data/"
# file_name = "submarine_articles.json"

# with open(f"{file_path}{file_name}",'w', encoding='utf-8') as fp:
#     fp.write(json.dumps(results))

The results contain the URL to the news items on the website. After inspecting a couple of pages, which information could be easily extracted from it

In [6]:
# Get HTML function
def get_HTML(url):
    # get data from server
    response = requests.get(url)
    html = response.content
    return html

In [7]:
# Beautiful soup function for subtitle
def extract_subtitle(HTML):
    soup = BeautifulSoup(HTML, "html.parser") # the html input and the parser name
    article = soup.find("article") # the tag that contains the article
    div_element = article.find("div", attrs={"data-gu-name": "standfirst"}) # the tag that can be found using an attribute
    if div_element is not None:
        target_element = div_element.find("p")
        return target_element.text
    else:
        return ""
    

In [8]:
# Beautiful soup function for body
def extract_body(HTML):
    soup = BeautifulSoup(HTML, "html.parser") # the html input and the parser name
    article = soup.find("article") # the tag that contains the article
    div_element = article.find("div", attrs={"id": "maincontent"}) # the tag that can be found using an attribute
    if div_element is not None:
        div_div_element = div_element.find("div")
        target_elements = div_element.find_all("p")
        result = ""
        for te in target_elements:
            result += te.text
        return result
    else:
        return ""

#### Clean/preprocess data

In [9]:
# Create a dataframe
df = pd.DataFrame(columns=["Date", "Section", "Title", "Subtitle", "Body"])
df

,Date,Section,Title,Subtitle,Body


In [10]:
# Populate the dataframe
for news in results:
    html = get_HTML(news["webUrl"])
    data = {"Date": news["webPublicationDate"], "Section": news["sectionName"], "Title": news["webTitle"], "Subtitle": extract_subtitle(html), "Body": extract_body(html)}
    df_to_append = pd.DataFrame([data])
    df = pd.concat([df,df_to_append], ignore_index=True)
df

,Date,Section,Title,Subtitle,Body
0,2021-10-31T19:27:45Z,World news,Macron accuses Australian PM of lying over sub...,French president criticises Scott Morrison and...,Emmanuel Macron has accused the Australian pri...
1,2021-10-07T16:39:54Z,World news,Aukus: French contractor ‘astonished’ at cance...,Head of Naval Group reiterates company’s ‘stup...,The head of the French defence contractor Nava...
2,2021-11-08T16:30:35Z,World news,Australia promises jobs to workers stranded by...,Defence industry minister Melissa Price tells ...,“Each and every” skilled shipbuilding worker a...
3,2021-10-29T17:06:48Z,World news,Biden admits to Macron the US was ‘clumsy’ in ...,American president moves to repair relationshi...,Joe Biden has moved to repair his damaged pers...
4,2021-10-01T09:00:58Z,World news,Fears Australia’s France submarine snub could ...,Opposition accuses Scott Morrison of failing ‘...,The postponement of trade talks between the Eu...
5,2021-10-28T04:39:00Z,World news,Australia’s foreign minister to meet French am...,Marise Payne says she regrets France’s ‘deep d...,Australia’s foreign minister will meet with th...
6,2021-09-29T02:50:32Z,World news,Former US navy secretary now Scott Morrison’s ...,"Prof Donald Winter, who advised the Australian...",A former US navy secretary who advised the Aus...
7,2021-09-16T11:32:12Z,World news,‘Stab in the back’: French fury as Australia s...,France’s foreign minister says move to buy nuc...,France has expressed fury over Australia’s sur...
8,2021-11-18T16:30:10Z,World news,‘Naughty guy’: top Chinese diplomat accuses Au...,"Exclusive: Acting ambassador to Australia, Wan...",A top Chinese diplomat has likened Australia t...
9,2021-09-20T01:11:38Z,World news,‘We felt fooled’: France still furious after A...,"‘Maybe we’re not friends,’ recalled ambassador...",French anger at the Morrison government’s deci...


What are the strategies to remove irrelevant articles? 

**Task**: Remove irrelevant articles before next step.

In [11]:
# for example:
df = df[df['Title'].str.lower().str.contains('submarine')]
df

,Date,Section,Title,Subtitle,Body
0,2021-10-31T19:27:45Z,World news,Macron accuses Australian PM of lying over sub...,French president criticises Scott Morrison and...,Emmanuel Macron has accused the Australian pri...
1,2021-10-07T16:39:54Z,World news,Aukus: French contractor ‘astonished’ at cance...,Head of Naval Group reiterates company’s ‘stup...,The head of the French defence contractor Nava...
2,2021-11-08T16:30:35Z,World news,Australia promises jobs to workers stranded by...,Defence industry minister Melissa Price tells ...,“Each and every” skilled shipbuilding worker a...
3,2021-10-29T17:06:48Z,World news,Biden admits to Macron the US was ‘clumsy’ in ...,American president moves to repair relationshi...,Joe Biden has moved to repair his damaged pers...
4,2021-10-01T09:00:58Z,World news,Fears Australia’s France submarine snub could ...,Opposition accuses Scott Morrison of failing ‘...,The postponement of trade talks between the Eu...
5,2021-10-28T04:39:00Z,World news,Australia’s foreign minister to meet French am...,Marise Payne says she regrets France’s ‘deep d...,Australia’s foreign minister will meet with th...
6,2021-09-29T02:50:32Z,World news,Former US navy secretary now Scott Morrison’s ...,"Prof Donald Winter, who advised the Australian...",A former US navy secretary who advised the Aus...
7,2021-09-16T11:32:12Z,World news,‘Stab in the back’: French fury as Australia s...,France’s foreign minister says move to buy nuc...,France has expressed fury over Australia’s sur...
8,2021-11-18T16:30:10Z,World news,‘Naughty guy’: top Chinese diplomat accuses Au...,"Exclusive: Acting ambassador to Australia, Wan...",A top Chinese diplomat has likened Australia t...
9,2021-09-20T01:11:38Z,World news,‘We felt fooled’: France still furious after A...,"‘Maybe we’re not friends,’ recalled ambassador...",French anger at the Morrison government’s deci...


### 3. Analysis

Information extraction?

#### Inspect the data

Read a few articles at random to get a feel for what is important to analyse.

#### One approach - a basic sentiment analysis that looks for positive and negative words in the text

In [12]:
# Define lists of positive and negative words
positive_words = ["good", "positive", "excellent", "success"] # add words you think are good indicators
negative_words = ["bad", "poor", "negative", "disappointing", "angry"]


# Function to calculate a basic sentiment score
def analyze_sentiment(article):
    positive_count = 0
    negative_count = 0
    
    # Convert article to lowercase and split into words
    words = article.lower().split()
    
    # Count occurrences of positive and negative words
    for word in words:
        if word in positive_words:
            positive_count += 1
        if word in negative_words:
            negative_count += 1
            
    # Compute sentiment score
    sentiment_score = positive_count - negative_count
    return sentiment_score



In [13]:
# Analyze the articles

# Create a list of article bodies from the df column
article_body_texts = df["Body"].tolist()

# Loop through and run the analyze_sentiment function on each
for article_text in article_body_texts:
    
    score = analyze_sentiment(article_text)

    # Print sentiment score
    print(f"Sentiment Score: {score}")

Sentiment Score: 0
Sentiment Score: 0
Sentiment Score: 0
Sentiment Score: 1
Sentiment Score: 0
Sentiment Score: 0
Sentiment Score: 0
Sentiment Score: -3
Sentiment Score: -1
Sentiment Score: 0
Sentiment Score: 0
Sentiment Score: 0
Sentiment Score: 0
Sentiment Score: 0
Sentiment Score: 0
Sentiment Score: 0
Sentiment Score: 1
Sentiment Score: 0
Sentiment Score: -1


#### Combine the scores and do some analysis

##### Tip: You could add them as a new column of your existing df


In [14]:
???

Object `?` not found.


### 4. Visualisation

In [15]:
???

Object `?` not found.


### 5. Insights

What might be some limitations of how you analysed the data?

#

# Scrape some data from the web to include as background in your final report

Use code similar to that in this week's lecture session to scrape some data from the web relevant to the submarine issue.

Suggestion: A list of current Australian submarines with their names and launch dates scraped from the web like the one at https://en.wikipedia.org/wiki/Collins-class_submarine#Submarines_in_class)